In [123]:
import pandas as pd

# load all_fouls_advanced.pkl
all_fouls_advanced = pd.read_pickle('all_fouls_advanced.pkl')

# load testing_fouls_advanced.pkl
testing_fouls_advanced = pd.read_pickle('testing_fouls_advanced.pkl')

# load model_lgbm_regressor.pkl
model_lgbm_regressor = pd.read_pickle('model_lgbm_regressor.pkl')

# load model_lgbm_classifier.pkl
model_lgbm_classifier = pd.read_pickle('model_lgbm_classifier.pkl')

# load all_the_features.pkl
all_the_features = pd.read_pickle('all_the_features.pkl')

# load all_the_features_test.pkl
all_the_features_test = pd.read_pickle('all_the_features_test.pkl')

# load all_the_labels.pkl
all_the_labels = pd.read_pickle('all_the_labels.pkl')

# load all_the_labels_test.pkl
all_the_labels_test = pd.read_pickle('all_the_labels_test.pkl')

In [124]:
# convert all_the_features which is a list to a dataframe. The name of columns are: ['minutes', 'score_difference', 'distance_to_goal', 'angle_to_goal', 'foul_count_player', 'foul_count_team', 'vaep_offensive', 'id']
all_the_features_df = pd.DataFrame(all_the_features, columns=['minutes', 'score_difference', 'distance_to_goal', 'angle_to_goal', 'foul_count_player', 'foul_count_team', 'vaep_offensive', 'id'])

# set id as index
all_the_features_df.set_index('id', inplace=True)

In [125]:
# convert all_the_features which is a list to a dataframe. The name of columns are: ['minutes', 'score_difference', 'distance_to_goal', 'angle_to_goal', 'foul_count_player', 'foul_count_team', 'vaep_offensive', 'id']
all_the_features_test_df = pd.DataFrame(all_the_features_test, columns=['minutes', 'score_difference', 'distance_to_goal', 'angle_to_goal', 'foul_count_player', 'foul_count_team', 'vaep_offensive', 'id'])

# set id as index
all_the_features_test_df.set_index('id', inplace=True)

In [126]:
# Set index of all_fouls_advanced to id
all_fouls_advanced.set_index('id', inplace=True)

# Set index of testing_fouls_advanced to id
testing_fouls_advanced.set_index('id', inplace=True)

In [127]:
from sklearn.preprocessing import StandardScaler

def getPredictions(feature_row):
    # get column values from feature_row and save it in a list
    feature_row = feature_row.values.tolist()

    # predict using model_lgbm_regressor
    prediction_regressor = model_lgbm_regressor.predict([feature_row])

    # predict using model_lgbm_classifier
    prediction_classifier = model_lgbm_classifier.predict([feature_row])

    return prediction_regressor, prediction_classifier

In [130]:
# scale all_the_features_df first 7 columns
scaler = StandardScaler()

# fit and transform all_the_features_df
all_the_features_df.iloc[:, :7] = scaler.fit_transform(all_the_features_df.iloc[:, :7])

# scale all_the_features_test_df first 7 columns
scaler = StandardScaler()

# fit and transform all_the_features_test_df
all_the_features_test_df.iloc[:, :7] = scaler.fit_transform(all_the_features_test_df.iloc[:, :7])

In [131]:
all_the_features_df.head()

,minutes,score_difference,distance_to_goal,angle_to_goal,foul_count_player,foul_count_team,vaep_offensive
id,,,,,,,
a40924e7-b5ca-4f23-9025-ac535d63dab8,-1.804209,-0.352575,0.562204,-0.705381,-0.481498,-1.090844,0.207948
24f49483-d437-40c2-bf5f-c6b3e5456d48,-1.613191,-0.352575,0.916698,0.828595,-0.481498,-0.761528,0.176081
681f5c57-ed69-4bae-82e7-6d97233fb907,0.144183,-0.352575,-1.528967,2.990464,-0.481498,-0.102896,0.173477
7ea7ec5e-388e-4b20-8ab5-d2359cb92a65,0.908258,-0.352575,1.798029,-0.033659,-0.481498,0.226419,-2.320992
6238286d-a363-4ed3-abab-f322e0571e2d,0.182387,-0.352575,0.682044,-0.990849,-0.481498,-0.761528,0.203386


In [132]:
all_the_features_test_df.head()

,minutes,score_difference,distance_to_goal,angle_to_goal,foul_count_player,foul_count_team,vaep_offensive
id,,,,,,,
9221da02-9c87-4f98-a8ea-de9994ad562b,-1.377034,0.614112,-1.072343,2.775660,-0.404798,-1.067673,0.215031
f2bd21f6-87c3-44c0-bb37-04e996db413e,-1.058653,1.821601,0.065315,0.059453,-0.404798,-1.067673,0.144903
8e51baec-6948-4285-86a1-73197126e265,-0.704897,0.614112,0.260115,-0.750153,-0.404798,-0.560311,0.160500
b711f529-182e-4ae3-8c9a-189b7697b2ea,-0.245013,0.614112,-0.477584,-0.656392,-0.404798,-0.560311,0.176348
ec46c71c-af4c-4d99-a854-e4ffc30460d6,0.073368,1.821601,-0.878762,1.159428,-0.404798,-0.052948,-0.370417


In [133]:
from tqdm import tqdm

# Add column prediction_regressor and prediction_classifier to all_the_features_df and all_the_features_test_df
all_the_features_df['prediction_regressor'] = 0
all_the_features_df['prediction_classifier'] = 0
all_the_features_df['actual_label'] = 100

# iterate through all_the_features_df and get predictions

i = 0
for index, row in tqdm(all_the_features_df.iterrows()):
    prediction_regressor, prediction_classifier = getPredictions(row[:7])
    all_the_features_df.loc[index, 'prediction_regressor'] = prediction_regressor
    all_the_features_df.loc[index, 'prediction_classifier'] = prediction_classifier
    
    # set actual_label using all_the_labels (which is a list)
    all_the_features_df.loc[index, 'actual_label'] = all_the_labels[i]
    i = i + 1

5572it [01:09, 80.44it/s] 


In [134]:
# Add column prediction_regressor and prediction_classifier to all_the_features_df and all_the_features_test_df
all_the_features_test_df['prediction_regressor'] = 0
all_the_features_test_df['prediction_classifier'] = 0
all_the_features_test_df['actual_label'] = 100

i = 0
# iterate through all_the_features_df and get predictions
for index, row in tqdm(all_the_features_test_df.iterrows()):
    prediction_regressor, prediction_classifier = getPredictions(row[:7])
    all_the_features_test_df.loc[index, 'prediction_regressor'] = prediction_regressor
    all_the_features_test_df.loc[index, 'prediction_classifier'] = prediction_classifier

    # set actual_label using all_the_labels (which is a list)
    all_the_features_test_df.loc[index, 'actual_label'] = all_the_labels_test[i]
    i = i + 1

757it [00:08, 84.24it/s] 


In [135]:
# save all_the_features_df as all_the_features_df.pkl
all_the_features_df.to_pickle('all_the_features_df.pkl')

# save all_the_features_test_df as all_the_features_test_df.pkl
all_the_features_test_df.to_pickle('all_the_features_test_df.pkl')

In [136]:
# combine all_the_features_df and all_the_features_test_df into a single dataframe
all_the_features_combined_df = pd.concat([all_the_features_df, all_the_features_test_df])



In [137]:
all_the_features_combined_df.head()

,minutes,score_difference,distance_to_goal,angle_to_goal,foul_count_player,foul_count_team,vaep_offensive,prediction_regressor,prediction_classifier,actual_label
id,,,,,,,,,,
a40924e7-b5ca-4f23-9025-ac535d63dab8,-1.804209,-0.352575,0.562204,-0.705381,-0.481498,-1.090844,0.207948,0.987373,1,2
24f49483-d437-40c2-bf5f-c6b3e5456d48,-1.613191,-0.352575,0.916698,0.828595,-0.481498,-0.761528,0.176081,0.931479,1,2
681f5c57-ed69-4bae-82e7-6d97233fb907,0.144183,-0.352575,-1.528967,2.990464,-0.481498,-0.102896,0.173477,0.920259,1,2
7ea7ec5e-388e-4b20-8ab5-d2359cb92a65,0.908258,-0.352575,1.798029,-0.033659,-0.481498,0.226419,-2.320992,0.914307,1,2
6238286d-a363-4ed3-abab-f322e0571e2d,0.182387,-0.352575,0.682044,-0.990849,-0.481498,-0.761528,0.203386,0.885819,1,2


In [138]:
# remove rows where all_the_features_combined_df['actual_label'] is 1.0
all_the_features_combined_df = all_the_features_combined_df[all_the_features_combined_df['actual_label'] != 1.0]

all_the_features_combined_df['actual_label'].value_counts()

2    3814
0    2460
Name: actual_label, dtype: int64

In [139]:
# convert all_the_features_combined_df['actual_label'] to int
all_the_features_combined_df['actual_label'] = all_the_features_combined_df['actual_label'].astype(int)

# change all 2 to 1 in all_the_features_combined_df['actual_label']
all_the_features_combined_df['actual_label'] = all_the_features_combined_df['actual_label'].replace(2, 1)

all_the_features_combined_df['actual_label'].value_counts()

1    3814
0    2460
Name: actual_label, dtype: int64

In [140]:
# save all_the_features_combined_df as all_the_features_combined_df.pkl
all_the_features_combined_df.to_pickle('all_the_features_combined_df.pkl')

In [1]:
all_the_features_combined_df.head(15)

NameError: name 'all_the_features_combined_df' is not defined